In [ ]:
import pandas as pd
import plotly.graph_objects as go

In [ ]:
# 工作路径
work_path = r'C:\Users\小小\PycharmProjects\第二次作业'
total_unemployment_file = '整体失业率.csv'
youth_unemployment_file = 'Unemployment Rate (Youth, ages 16–24).csv'

In [ ]:
# 读取整体失业率数据
try:
    df_total = pd.read_csv(f'{work_path}\\{total_unemployment_file}')
except FileNotFoundError:
    print(f"File {total_unemployment_file} not found. Please check the file name and path.")
    raise
else:
    # 提取美国2000 - 2023年的总体失业率数据
    us_total = df_total[(df_total['Country Name'] == 'United States') & (df_total['Year'].between(2000, 2023))]
    us_total = us_total[['Year', 'Value']].rename(columns={'Value': 'Total_Unemployment_Rate'})

In [ ]:
# 读取青年失业率数据
try:
    df_youth = pd.read_csv(f'{work_path}\\{youth_unemployment_file}')
except FileNotFoundError:
    print(f"File {youth_unemployment_file} not found. Please check the file name and path.")
    raise
else:
    # 提取美国2000 - 2023年的青年失业率数据
    df_youth['Year'] = pd.to_datetime(df_youth['observation_date']).dt.year
    us_youth = df_youth[df_youth['Year'].between(2000, 2023)]
    us_youth = us_youth[['Year', 'LNS14024887']].rename(columns={'LNS14024887': 'Youth_Unemployment_Rate'})

In [ ]:
# 合并数据
combined_data = pd.merge(us_total, us_youth, on='Year')

In [ ]:
# 创建交互式趋势图
fig = go.Figure()

In [ ]:
# 添加总体失业率数据
fig.add_trace(go.Scatter(
    x=combined_data['Year'],
    y=combined_data['Total_Unemployment_Rate'],
    mode='lines+markers',
    name='Total Unemployment Rate',
    marker=dict(symbol='circle')
))

In [ ]:
# 添加青年失业率数据
fig.add_trace(go.Scatter(
    x=combined_data['Year'],
    y=combined_data['Youth_Unemployment_Rate'],
    mode='lines+markers',
    name='Youth Unemployment Rate (16 - 24 years old)',
    marker=dict(symbol='square')
))

In [ ]:
# 突出显示经济危机时期
crisis_years = [2001, 2008, 2020]
for year in crisis_years:
    crisis_total = combined_data[combined_data['Year'] == year]['Total_Unemployment_Rate'].values[0]
    crisis_youth = combined_data[combined_data['Year'] == year]['Youth_Unemployment_Rate'].values[0]
    fig.add_trace(go.Scatter(
        x=[year],
        y=[crisis_total],
        mode='markers',
        name=f'Total Unemployment in {year}',
        marker=dict(symbol='circle', color='red', size=10)
    ))
    fig.add_trace(go.Scatter(
        x=[year],
        y=[crisis_youth],
        mode='markers',
        name=f'Youth Unemployment in {year}',
        marker=dict(symbol='square', color='red', size=10)
    ))
    fig.add_annotation(
        x=year,
        y=crisis_total + 0.3,
        text=f'2001 Dot-Com Bubble Burst' if year == 2001 else '2008 Financial Crisis' if year == 2008 else 'COVID-19 Pandemic',
        showarrow=True,
        arrowhead=1,
        ax=0,
        ay=-40,
        font=dict(color='red')
    )
    fig.add_annotation(
        x=year,
        y=crisis_youth - 0.3,
        text=f'2001 Dot-Com Bubble Burst' if year == 2001 else '2008 Financial Crisis' if year == 2008 else 'COVID-19 Pandemic',
        showarrow=True,
        arrowhead=1,
        ax=0,
        ay=40,
        font=dict(color='red')
    )

In [ ]:
# 设置图表布局
fig.update_layout(
    title='Unemployment Rate Trend in the United States from 2000 - 2023',
    xaxis_title='Year',
    yaxis_title='Unemployment Rate (%)',
    hovermode='x unified',
    xaxis=dict(
        tickmode='array',
        tickvals=combined_data['Year'][::2],
        tickangle=45
    ),
    legend=dict(x=0.7, y=1.1, xanchor='left', yanchor='top'),
    plot_bgcolor='white',
    showlegend=True
)

In [ ]:
# 保存为 HTML 文件
fig.write_html('unemployment_rate_trend_interactive.html')
print("Interactive HTML file saved as unemployment_rate_trend_interactive.html")